[![Open In Colab](https://google.com)](https://google.comhamagami/IS2026/blob/main/Demo_31_MCTS_LearnedModel.ipynb)

# Demo 31: Tree Search: Known and Learned Models / 既知モデルと学習モデルによる木探索

Compare planning using known rewards and rewards estimated from experience in a simplified two-step tree. / 2手先までの簡略木で、既知の報酬と経験から推定した報酬を用いる探索を比較する。

**How to use / 使い方：** Run the cells from top to bottom, then change the controls and compare the figures. / セルを上から順に実行し、操作項目を変えて図を比較する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
from collections import defaultdict
ACTIONS_M=['Left','Center','Right']
# 2つの行動の組に依存する隠れた真の報酬 / Hidden oracle rewards for pairs of actions.
TRUE=np.array([[.35,.85,.22],[.52,.41,.72],[.18,.65,1.05]])
NOISE=.1

def collect_experience(n=50,seed=0):
    rng=np.random.default_rng(seed);a=rng.integers(0,3,size=n);b=rng.integers(0,3,size=n)
    r=TRUE[a,b]+rng.normal(0,NOISE,n)
    return a,b,r

def learn_reward_model(n=50,seed=0,prior=.1):
    aa,bb,rr=collect_experience(n,seed);count=np.zeros((3,3));summ=np.zeros((3,3))
    for a,b,r in zip(aa,bb,rr):count[a,b]+=1;summ[a,b]+=r
    # 未観測系列は真の報酬ではなく中立の事前値へ平滑化 / Smooth unseen sequences toward a neutral prior, not true reward.
    return (summ+prior*.5)/(count+prior),count

class SearchNode:
    def __init__(self,seq=()):self.seq=seq;self.visits=0;self.sum_reward=0.;self.children={}

def mcts(reward_table,budget=100,seed=0):
    rng=np.random.default_rng(seed);root=SearchNode(())
    for _ in range(budget):
        node=root;path=[node]
        while len(node.seq)<2:
            if len(node.children)<3:
                unseen=[a for a in range(3) if a not in node.children]
                a=int(rng.choice(unseen));node.children[a]=SearchNode(node.seq+(a,));node=node.children[a];path.append(node)
                break
            # UCB1で未知分岐の探索と既知価値の利用を両立 / UCB1 balances exploration and exploitation.
            a=max(node.children,key=lambda a:(node.children[a].sum_reward/max(1,node.children[a].visits)+1.2*np.sqrt(np.log(node.visits+1)/max(1,node.children[a].visits))))
            node=node.children[a];path.append(node)
        seq=node.seq
        while len(seq)<2:seq=seq+(int(rng.integers(3)),)
        outcome=float(reward_table[seq[0],seq[1]])
        for node in path:node.visits+=1;node.sum_reward+=outcome
    stats={a:(root.children[a].visits,root.children[a].sum_reward/max(1,root.children[a].visits)) if a in root.children else (0,0.) for a in range(3)}
    selected=max(stats,key=lambda a:stats[a][0])
    return selected,stats,root

m,counts=learn_reward_model(100,0);pick,stats,root=mcts(m,100,0)
assert sum(v[0] for v in stats.values())==100

In [ ]:
experiences=widgets.IntSlider(value=40,min=5,max=250,step=5,description='Data / 経験数',continuous_update=False)
budget=widgets.IntSlider(value=90,min=9,max=500,step=10,description='MCTS / 探索回数',continuous_update=False)
seed=widgets.IntSlider(value=4,min=0,max=100,description='Seed / 乱数')
model=widgets.ToggleButtons(options=['Known / 既知モデル','Learned / 学習モデル'],description='Planning / 計画')
button=widgets.Button(description='Search / 探索',button_style='primary');out=widgets.Output()
def draw_search(_):
    with out:
        clear_output(wait=True)
        learned,cnt=learn_reward_model(experiences.value,seed.value)
        estimates=TRUE if model.value.startswith('Known') else learned
        chosen,stats,tree=mcts(estimates,budget.value,seed.value)
        fig,ax=plt.subplots(1,3,figsize=(14.5,3.9))
        ax[0].imshow(TRUE,vmin=0,vmax=1.15,cmap='YlGnBu');ax[0].set(title='True rewards / 真の報酬',xticks=range(3),yticks=range(3),xticklabels=ACTIONS_M,yticklabels=ACTIONS_M,xlabel='Action 2 / 2手目',ylabel='Action 1 / 1手目')
        for i in range(3):
            for j in range(3):ax[0].text(j,i,f'{TRUE[i,j]:.2f}',ha='center',va='center')
        ax[1].imshow(learned,vmin=0,vmax=1.15,cmap='YlGnBu');ax[1].set(title=f'Learned rewards ({experiences.value} samples)',xticks=range(3),yticks=range(3),xticklabels=ACTIONS_M,yticklabels=ACTIONS_M,xlabel='Action 2 / 2手目',ylabel='Action 1 / 1手目')
        for i in range(3):
            for j in range(3):ax[1].text(j,i,f'{learned[i,j]:.2f}',ha='center',va='center')
        visits=[stats[i][0] for i in range(3)];ax[2].bar(ACTIONS_M,visits,color=['#9ACAF4' if i!=chosen else '#F8B6CA' for i in range(3)]);ax[2].set(title='MCTS visits / 探索回数',ylabel='Visits / 訪問数',xlabel='Action 1 / 1手目')
        plt.tight_layout();plt.show()
        print('Selected action / 選んだ初手:',ACTIONS_M[chosen])
        print('Visits and simulated mean / 訪問回数と仮想平均収益:',{ACTIONS_M[i]:(stats[i][0],round(stats[i][1],3)) for i in range(3)})
        print('True best first action / 真の最適初手:',ACTIONS_M[int(np.argmax(TRUE.max(axis=1)))])
        if model.value.startswith('Learned'):print('Observed sequences / 観測した行動系列:',int((cnt>0).sum()),'/ 9; model estimates may be inaccurate / 推定誤差の可能性あり。')
button.on_click(draw_search)
display(widgets.VBox([widgets.HBox([experiences,budget,seed]),model,button,out]));draw_search(None)

## Relation to AlphaZero and MuZero / AlphaZero・MuZeroとの関係

- **AlphaZero:** receives the game rules (transition dynamics), learns policy/value from self-play, and uses tree search. / **AlphaZero：** ゲームのルール（遷移）を与えられ、自己対戦で方策・価値を学び、木探索を使う。
- **MuZero:** learns task-relevant latent dynamics, rewards, values and policy from experience, without a complete given transition model. / **MuZero：** 完全な状態遷移モデルを外部から与えず、経験から意思決定に必要な潜在ダイナミクス・報酬・価値・方策を学ぶ。
- This **simplified demonstration** compares search with known versus learned reward estimates. It is not a full AlphaZero/MuZero network. / この**簡略デモ**は既知と学習済みの報酬予測による探索を比べるもので、AlphaZero/MuZeroの完全な実装ではない。

## Try it yourself / 実験課題

1. Increase the experience count from 10 to 150 and observe the learned estimates. / 経験数を10から150に増やし、学習モデルの予測の変化を観察する。
2. Increase search simulations and see whether the chosen first action changes. / 探索回数を増やし、選ばれる初手が変わるか確かめる。
3. Distinguish **model error** from **insufficient search**. / **モデル誤差**と**探索不足**を区別する。